# 하나은행 환율정보 수집하고 DB에 자동으로 저장하기_수집자동화

In [1]:
import requests
import pandas as pd
import time
from datetime import datetime
from io import StringIO

In [2]:
# 하나은행 환율 수집 함수
def exrate_get(ymd_dash, ymd):
    url = "https://www.kebhana.com/cms/rate/wpfxd651_01i_01.do" 
    payload = dict(ajax="true", tmpInqStrDt=ymd_dash, pbldDvCd="0", inqStrDt=ymd, 
                   inqKindCd="1", requestTarget="searchContentDiv")
    r = requests.get(url, params = payload)
    # 웹페이지 HTML에서 표를 전부 읽은 뒤, 첫 번째 표를 pandas DataFrame으로 반환
    df = pd.read_html(StringIO(r.text))[0]
    # DataFrame의 맨 앞(0번째 열)에 ‘날짜’라는 새 컬럼을 추가하고, 모든 행에 같은 날짜 값을 넣는 코드
    df.insert(0, '날짜', ymd_dash)
    return df

# 날짜 생성하기

In [7]:
result = []
for date in list(pd.date_range("2025-01-28", "2026-01-28", freq ='B')):
    ymd_dash = date.strftime("%Y-%m-%d")
    ymd = date.strftime("%Y%m%d")
    data = exrate_get(ymd_dash, ymd)
    result.append(data)
final_result = pd.concat(result)
final_result

날짜            통화       현찰                              송금  \
                          통화     사실 때            파실 때            보낼 때   
                          통화       환율 Spread       환율 Spread     보낼 때   
0   2025-01-28        미국 USD  1471.30   1.75  1420.70   1.75  1460.10   
1   2025-01-28  일본 JPY (100)   953.90   1.75   921.10   1.75   946.68   
2   2025-01-28        유로 EUR  1531.92   1.99  1472.14   1.99  1517.05   
3   2025-01-28        중국 CNY   208.15   5.00   188.33   5.00   200.22   
4   2025-01-28        홍콩 HKD   189.22   1.97   181.92   1.97   187.42   
..         ...           ...      ...    ...      ...    ...      ...   
53  2026-01-28       리비아 LYD     0.00   0.00     0.00   0.00   231.06   
54  2026-01-28      루마니아 RON     0.00   0.00     0.00   0.00   340.08   
55  2026-01-28       미얀마 MMK     0.00   0.00     0.00   0.00     0.68   
56  2026-01-28     에티오피아 ETB     0.00   0.00     0.00   0.00     9.33   
57  2026-01-28    우즈베키스탄 UZS     0.00   0.00     0.00   0.00     0.12   

            외화 수표 파실때   매매 기준율     환가 료율  미화 환산율  
       받을 때 외화 수표 파실때   매매 기준율     환가 료율  미화 환산율  
       받을 때 외화 수표 파실때   매매 기준율     환가 료율  미화 환산율  
0   1431.90   1429.43  1446.00   6.16064  1.0000  
1    928.32    927.73   937.50   2.56227  0.6483  
2   1487.01   1485.03  1502.03   4.75100  1.0387  
3    196.26      0.00   198.24   4.36652  0.1371  
4    183.72    183.44   185.57   6.23166  0.1283  
..      ...       ...      ...       ...     ...  
53   225.60      0.00   228.33   2.97500  0.1593  
54   332.68      0.00   336.38   7.77066  0.2347  
55     0.68      0.00     0.68   1.97500  0.0005  
56     9.11      0.00     9.22   2.97500  0.0064  
57     0.12      0.00     0.12  17.47500  0.0001  

[15196 rows x 12 columns]

In [ ]:
final_result.columns

# 멀티인덱스 컬럼명 단순화하기

In [ ]:
for col in final_result.columns:
    print(col)

# 1일 데이터 수집 후 바로 db 저장하기

In [ ]:
result = []
for date in list(pd.date_range("2025-01-28", "2026-01-28", freq ='B')):
    ymd_dash = date.strftime("%Y-%m-%d")
    ymd = date.strftime("%Y%m%d")
    data = exrate_get(ymd_dash, ymd)
    
    new.cols_dict = flatten_cols(data)

오늘 날짜로 데이터 수집하기

In [8]:
today = datetime.today()
datetime.today()

datetime.datetime(2026, 1, 29, 11, 17, 56, 479511)

In [ ]:
def new_cols(df):
    new_cols = []
    for col in df.columns:

        if col[0] == col[1] == col[2]:
            new_cols.append(col[0].replace(" ", "_"))
        elif col[0] != col[1] != col[2]:
            new_cols.append("_".join(col).replace(" ", "_"))
        else:
            new_cols.append("_".join(col[:2]).replace(" ", "_"))
    return new_cols